# 📚 만화 번역기 — 원클릭 실행 노트북

## 사용법
1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 좌측 🔑(보안 비밀)에 두 개 등록: `GEMINI_API_KEY`, `GITHUB_TOKEN`
3. **런타임 → 모두 실행** (또는 셀을 위에서부터 차례로)
4. 마지막 셀에서 만화 이미지를 업로드하면 번역 결과가 나옵니다

> 첫 실행은 라이브러리·모델 다운로드로 수 분 걸립니다. 재시작은 필요 없습니다.

In [ ]:
# ① 코드 + 라이브러리 + 한국어 폰트 설치
import os
from google.colab import userdata

if not os.path.exists('/content/manga-translator'):
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/auddpfkql-spec/Manga-translation-web-app-planning.git /content/manga-translator
%cd /content/manga-translator
!git pull -q

!pip install -q -r requirements.txt

# 한국어 폰트 (없으면 결과가 □□□ 두부로 나옴)
!apt-get -qq install -y fonts-nanum > /dev/null
!cp /usr/share/fonts/truetype/nanum/NanumGothic.ttf fonts/ 2>/dev/null || true
print('✅ ① 설치 완료')

In [ ]:
# ② CUDA torch 복구 — nvidia 런타임 라이브러리(libcusparseLt 등)까지 함께 설치.
#    (--no-deps 를 쓰면 그 라이브러리가 빠져 import 시 libcusparseLt.so.0 에러가 난다)
!pip install -q torch torchvision --force-reinstall --index-url https://download.pytorch.org/whl/cu124
print('✅ ② torch 복구 완료')

In [ ]:
# ③ 서버 실행 (여기서 torch를 처음 import → 위에서 깐 버전이 적용됨, 재시작 불필요)
import os, sys, threading, time
os.chdir('/content/manga-translator')
sys.path.insert(0, '/content/manga-translator')

from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

import torch
print('CUDA 사용 가능:', torch.cuda.is_available())   # True 여야 함
if not torch.cuda.is_available():
    print('⚠️ CUDA False — 런타임이 T4 GPU인지 확인하고 이 셀만 다시 실행하세요.')

import nest_asyncio, uvicorn
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True,
).start()
time.sleep(3)
print('✅ ③ 서버 실행됨')

In [ ]:
# ④ 만화 이미지 업로드 → 번역 → 결과 이미지 표시
#    (LaMa 인페인팅 모델이 아직 없어 inpainter=none: 원문 위에 한국어를 바로 얹음)
from google.colab import files
import requests, base64, io
from PIL import Image

up = files.upload()                      # 일본어 만화 이미지 선택
fname = list(up.keys())[0]

with open(fname, 'rb') as f:
    r = requests.post(
        'http://localhost:8000/api/translate',
        files={'file': f},
        data={'options': '{"source_lang":"ja","inpaint":{"inpainter":"none"}}'},
        timeout=600,                     # 첫 요청은 모델 다운로드로 수 분
    )

print('상태코드:', r.status_code)
if r.status_code != 200:
    print(r.text[:800])
else:
    blocks = r.json()['blocks']
    print('✅ 성공! 감지된 텍스트 블록:', len(blocks))
    for b in blocks[:10]:
        print('  -', b.get('original_text'), '→', b.get('translated_text'))
    img_b64 = r.json()['result_image'].split(',', 1)[1]
    display(Image.open(io.BytesIO(base64.b64decode(img_b64))))